In [20]:
import json
import pandas as pd

In [21]:
filename = "data/predictions/2026_week_06.json"

In [22]:
def load_json_file(filename):
    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)

    results = data["results"]
    comments = data["comments"]

    print(f"Loaded {len(results)} results from {filename}")
    print(f"Loaded {len(comments)} comments from {filename}")

    return results, comments

In [23]:
results, comments = load_json_file(filename)

Loaded 10 results from data/predictions/2026_week_06.json
Loaded 64 comments from data/predictions/2026_week_06.json


In [24]:
def predictions_to_dataframe(predictions):
    rows = []

    for comment in predictions:
        comment_data = {
            "comment_id": comment["comment_id"],
            "author": comment["author"],
            "published_at": comment["published_at"],
        }

        for fixture, pred in comment["predictions"].items():
            rows.append({
                **comment_data,
                **pred,
            })

    df_predictions = pd.DataFrame(rows)
    print(f"Loaded {len(df_predictions)} results into DataFrame")
    return df_predictions

In [25]:
def results_to_dataframe(results):
    result_keys = [
    "fixture",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
    "winner",
    ]

    df_results = pd.DataFrame([
        {key: result.get(key) for key in result_keys}
        for result in results.values()
    ])
    print(f"Loaded {len(df_results)} results into DataFrame")
    return df_results

In [26]:
def save_dataframe_to_csv(df, filename):
    df.to_csv(filename, index=False)
    print(f"Saved DataFrame to {filename}")

In [27]:
def save_dfs_to_csv(results_df, predictions_df, results_filename, predictions_filename):
    save_dataframe_to_csv(results_df, results_filename)
    save_dataframe_to_csv(predictions_df, predictions_filename)

In [28]:
# df to csv
pred_df = predictions_to_dataframe(comments)
pred_df.to_csv("data/predictions/2026_week_06_predictions.csv", index=False)

Loaded 640 results into DataFrame


In [29]:
# df to csv
results_df = results_to_dataframe(results)
results_df.to_csv("data/predictions/2026_week_06_results.csv", index=False)

Loaded 10 results into DataFrame


In [30]:
def main_function(filename,debug=True):
    results, predictions = load_json_file(filename)
    results_df = results_to_dataframe(results)
    predictions_df = predictions_to_dataframe(predictions)
    if debug:
        save_dfs_to_csv(results_df, predictions_df, "results.csv", "predictions.csv")
    

In [35]:
# Load test dataset csv
results_df = pd.read_csv("data/predictions/2026_week_06_results.csv")
pred_df = pd.read_csv("data/predictions/2026_week_06_predictions.csv")


In [37]:
print(pred_df.head(1))

                   comment_id             author          published_at  \
0  UgxqMhX0HsWgFA_bHct4AaABAg  @TheIrishGuyExtra  2026-02-06T19:22:08Z   

                     fixture  found prediction home_team          away_team  \
0  Leeds - Nottingham_Forest   True        0-0     Leeds  Nottingham_Forest   

   home_score  away_score winner  
0         0.0         0.0   draw  


In [39]:
# list unique authors
unique_authors = pred_df['author'].unique()

# empty dictionary to store points for each author

author_points = {}



for author in unique_authors:
    points = 0
    # compare matches for each author with the results dataframe, use fixture as the key of comparison and increase counter by 5 if winner is the same, and by 10 if home_score and away_score are also the same
    author_predictions = pred_df[pred_df['author'] == author]

    for index, row in author_predictions.iterrows():
        #print(f"Author: {author}, Fixture: {row['fixture']}, Predicted Winner: {row['winner']}, Predicted Home Score: {row['home_score']}, Predicted Away Score: {row['away_score']}")
        # compare with results dataframe
        # match fixtures in results and current row
        result_row = results_df[results_df['fixture'] == row['fixture']]
        if not result_row.empty:
            result_row = result_row.iloc[0]
            if row['winner'] == result_row['winner']:
                points += 5
                if row['home_score'] == result_row['home_score'] and row['away_score'] == result_row['away_score']:
                    points += 5

    # save points for each author in the dictionary
    author_points[author] = points
    print(f"Author: {author}, Points: {points}")

# save author points to csv
author_points_df = pd.DataFrame(list(author_points.items()), columns=['author', 'points'])
author_points_df.to_csv("data/predictions/2026_week_06_author_points.csv", index=False)
# print(f"Unique authors: {len(unique_authors)}")

Author: @TheIrishGuyExtra, Points: 10
Author: @GaelicGuy24, Points: 25
Author: @ManchesterUnited20fc, Points: 10
Author: @evelynmungau9319, Points: 15
Author: @vandy5113, Points: 30
Author: @stibbz25, Points: 30
Author: @VikkiOShea, Points: 40
Author: @PR-li3xm, Points: 35
Author: @stephensixsmith, Points: 25
Author: @spamcrap-g6o, Points: 40
Author: @codyashton1151, Points: 25
Author: @mikejackson19828, Points: 25
Author: @LurDurr, Points: 5
Author: @łłłłłłłłłłł-l16, Points: 35
Author: @krastiopetov8796, Points: 25
Author: @RyanH2005, Points: 50
Author: @Thatguyofir, Points: 5
Author: @ScotlandSerenity, Points: 15
Author: @coffeegoblin, Points: 50
Author: @CrankyLanky, Points: 10
Author: @leomccarron9133, Points: 25
Author: @Gnadeeer, Points: 5
Author: @therizah_25, Points: 10
Author: @IAmTheOneAndOnly-i8o, Points: 30
Author: @callummusik5044, Points: 5
Author: @jinkotadia8785, Points: 45
Author: @PPenguin17, Points: 5
Author: @Maidrhap, Points: 30
Author: @blero473, Points: 40
Author